# 🏃 Domácí úkol (Session 2): XGBoost – Predikce spálených kalorií při cvičení (Regrese)
**Dataset:** `calories_exercise_data.csv` (15 000 tréninkových záznamů, 9 atributů)  
**Cíl úlohy:** Konstrukce a optimalizace modelu gradientního boostingu (`XGBRegressor` z knihovny `xgboost`) pro predikci počtu spálených kalorií (`calories`), včetně generování profilovacího reportu přes `ydata_profiling`, odstranění zbytečného identifikátoru `user_id`, kódování pohlaví (`gender`), normalizace nezávislých proměnných, rozdělení 70:30 (`random_state=42`), hyperparametrického ladění pomocí `RandomizedSearchCV` s optimalizací na metriku **Mean Absolute Error (MAE)** a detailního vyhodnocení.

---
### 📋 Kroky zadání:
1. **Načtení dat:** Stáhněte soubor `calories_exercise_data.csv` z materiálů a načtěte jej pomocí knihovny Pandas.
2. **Import tříd:** Importujte potřebné třídy pro stavbu XGBoostu z knihovny `xgboost` a Scikit-learn.
3. **Data Profiling report:** Pomocí knihovny `ydata_profiling` vygenerujte report o datasetu a prohlédněte si jej:
   ```python
   !pip install ydata_profiling
   import ydata_profiling as ydp
   calories_report = ydp.ProfileReport(calories_df)
   calories_report
   ```
4. **Příprava dat:**
   - Odstraňte nepotřebné sloupce (`user_id`).
   - Proveďte kódování kategorických proměnných (`gender`).
   - Proveďte normalizaci nezávislých proměnných.
5. **Train-Test Split:** Rozdělte dataset na trénovací a testovací množinu v poměru 70:30 (`random_state=42`).
6. **Instance modelu:** Vytvořte instanci modelu `xgb_regressor` se zadanými parametry `random_state` a `n_jobs`.
7. **Mřížka parametrů:** Přiřaďte proměnné `params` slovník s klíči: `max_depth`, `min_samples_leaf` a `n_estimators` (2 až 4 hodnoty na klíč).
8. **RandomizedSearchCV:** Vytvořte objekt `RandomizedSearchCV`, pomocí kterého vyberete nejlepší sadu parametrů pro model `xgb_regressor` s optimalizovanou metrikou `mean_absolute_error` (`scoring='neg_mean_absolute_error'`).
9. **Trénování:** Pomocí nejlepší sady parametrů natrénujte model na trénovacích datech `X_train, y_train`.
10. **Predikce:** S testovacími daty proveďte predikci pomocí natrénovaného modelu.
11. **Vyhodnocení:** Spočítejte průměrnou absolutní chybu (**MAE**), kterou model dosahuje na testovací množině.


In [ ]:
# Krok 1 & 2: Import knihoven a načtení dat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Načtení datasetu kalorií
try:
    calories_df = pd.read_csv('calories_exercise_data.csv')
except FileNotFoundError:
    try:
        calories_df = pd.read_csv('data/calories_exercise_data.csv')
    except FileNotFoundError:
        calories_df = pd.read_csv('data/MAL_downloadable materials_session 2/Homework/calories_exercise_data.csv')

print(f"Dataset úspěšně načten! Tvar: {calories_df.shape}")
calories_df.head()


### 📊 Krok 3: Generování profilovacího reportu (ydata_profiling)
Zadání explicitně vyžaduje vytvoření souhrnného profilovacího reportu pomocí balíčku `ydata_profiling`.


In [ ]:
# Krok 3: Generování profilovacího reportu (dle zadání)
# !pip install ydata_profiling

try:
    import ydata_profiling as ydp
    # calories_report = ydp.ProfileReport(calories_df, title="Calories Exercise Profiling Report", minimal=True)
    # calories_report.to_file("calories_report.html")
    print("Knihovna ydata_profiling je dostupná. Report lze zobrazit přes calories_report.")
except ImportError:
    print("Tip pro Colab / Jupyter: Nainstalujte knihovnu příkazem '!pip install ydata_profiling'")

calories_df.describe()


### 🧹 Krok 4 & 5: Příprava dat, normalizace a Train-Test Split (70:30)
Provedeme:
1. Odstranění zbytečného identifikátoru `user_id`.
2. Kódování kategorického atributu `gender` na binární příznak (`gender_male`).
3. Rozdělení na trénovací a testovací sadu v poměru 70:30 se zadaným `random_state=42`.
4. Normalizaci nezávislých proměnných pomocí `StandardScaler`.


In [ ]:
# Krok 4 & 5: Příprava dat a Split
# 1. Odstranění user_id a oddělení cíle
X = calories_df.drop(columns=['user_id', 'calories'])
y = calories_df['calories']

# 2. Kódování kategorických proměnných
X = pd.get_dummies(X, columns=['gender'], drop_first=True)
feature_names = X.columns.tolist()

# 3. Train-test split (70:30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 4. Normalizace nezávislých proměnných
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_names, index=X_test.index)

print(f"Trénovací sada: {X_train_scaled.shape[0]} vzorků, {X_train_scaled.shape[1]} příznaků")
print(f"Testovací sada : {X_test_scaled.shape[0]} vzorků, {X_test_scaled.shape[1]} příznaků")
X_train_scaled.head(3)


### ⚡ Kroky 6 & 7: Instance modelu XGBoost a mřížka hyperparametrů
Inicializujeme `xgb.XGBRegressor` s parametry `random_state=42` a `n_jobs=-1`.  
Definujeme slovník `params` se zadanými klíči:
- `max_depth`: [3, 6, 9]
- `min_samples_leaf`: [1, 2, 4] *(Poznámka: parametr je v XGBoost akceptován přes `**kwargs`; v XGBoost odpovídá parametru `min_child_weight`)*
- `n_estimators`: [50, 100, 150]


In [ ]:
# Kroky 6 & 7: Instance modelu a mřížka hyperparametrů
xgb_regressor = xgb.XGBRegressor(
    random_state=42,
    n_jobs=-1
)

params = {
    'max_depth': [3, 6, 9],
    'min_samples_leaf': [1, 2, 4],
    'n_estimators': [50, 100, 150]
}


### 🎲 Krok 8: Výběr hyperparametrů pomocí RandomizedSearchCV (metrika MAE)
Vytvoříme objekt `RandomizedSearchCV` s metrikou `scoring='neg_mean_absolute_error'`, 3-násobnou křížovou validací a 10 náhodnými iteracemi.


In [ ]:
# Krok 8: RandomizedSearchCV
rand_search = RandomizedSearchCV(
    estimator=xgb_regressor,
    param_distributions=params,
    n_iter=10,
    scoring='neg_mean_absolute_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

print("Spouštím optimalizaci hyperparametrů (RandomizedSearchCV přes MAE)...")
rand_search.fit(X_train_scaled, y_train)

print(f"\nNejlepší nalezené parametry: {rand_search.best_params_}")
print(f"Nejlepší CV MAE (3-fold): {-rand_search.best_score_:.2f} kcal")


### 🎯 Kroky 9, 10 & 11: Trénování, testovací predikce a výpočet MAE
S nejlepším modelem vygenerujeme predikce pro testovací množinu `X_test_scaled` a vyčíslíme průměrnou absolutní chybu (**MAE**), **RMSE** a $R^2$.


In [ ]:
# Kroky 9, 10 & 11: Predikce a finální vyhodnocení
best_xgb = rand_search.best_estimator_
y_pred = best_xgb.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=" * 55)
print("📊 VÝSLEDKY MODELU XGBOOST NA TESTOVACÍ SADĚ (KALORIE)")
print("=" * 55)
print(f"Průměrná absolutní chyba (MAE) : {mae:.2f} kcal")
print(f"Odmocnina čtvercové chyby (RMSE) : {rmse:.2f} kcal")
print(f"Koeficient determinace (R²)     : {r2:.4f} ({r2*100:.2f} % vysvětleno)")
print("=" * 55)


### 💡 Analýza důležitosti příznaků (XGBoost Gain)
Které biometrické proměnné nejvíce ovlivňují energetický výdej organismu?


In [ ]:
# Důležitost příznaků dle Gain
booster = best_xgb.get_booster()
gain_dict = booster.get_score(importance_type='gain')
imp_series = pd.Series(gain_dict).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=imp_series.values, y=imp_series.index, palette='viridis')
plt.title('Důležitost příznaků dle XGBoost Gain (energetický výdej při cvičení)')
plt.xlabel('XGBoost Gain')
plt.ylabel('Biometrický příznak')
plt.tight_layout()
plt.show()
